# FedQLeaf-Net — Full Working Reference Implementation

**FedQLeaf-Net: A Federated Quantum-Enhanced Spatial-Frequency Framework for Edge-Deployable Leaf Disease Diagnosis**

This notebook implements the main computational pipeline described in the supplied manuscript:

1. Dataset discovery and train/validation/test splitting
2. Client formation with Dirichlet non-IID partitioning
3. Client-local preprocessing with GrabCut + CLAHE
4. Learnable spatial-frequency representation with wavelet-style decomposition
5. Spatial additive-attention branch
6. Spatial-Frequency Adaptive Gating (SF-Gate)
7. Sparse Mixture-of-Experts routing
8. 128-D bottleneck representation
9. 4-qubit angle-encoded variational quantum circuit
10. Quantum Interaction Fusion (QIF)
11. Three-head early-exit classifier
12. Gate-aware entropy exit rule
13. Fake-quantization during training and post-training 4/6/8-bit weight quantization
14. Federated weighted parameter aggregation
15. Accuracy, precision, recall, F1, confusion matrix and ROC-AUC
16. Communication-cost and client-heterogeneity diagnostics
17. Single-image inference with selected exit head

The manuscript specifies 100 communication rounds, 5 local epochs, 12 clients, Dirichlet α=0.3, 224×224 input, AdamW and batch size 32. For practical Colab execution, the notebook exposes a **DEMO_MODE** switch; set it to `False` for the manuscript-scale configuration. The notebook never fabricates final experimental metrics: all reported evaluation metrics are calculated from the selected dataset split.

In [ ]:
# 1. Installation — run once in Colab
# The notebook is designed for Google Colab / Python 3.10+.

!pip -q install pennylane opencv-python-headless scikit-learn pandas matplotlib seaborn pillow tqdm

import os, sys, math, random, time, copy, json, warnings, hashlib
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models

import pennylane as qml

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    classification_report, roc_auc_score, roc_curve, balanced_accuracy_score
)

print("PyTorch:", torch.__version__)
print("PennyLane:", qml.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 2. Configuration
# Change DATA_ROOT to the directory containing class folders.
# Example:
# /content/PlantVillage/
#   Apple___Apple_scab/
#   Apple___Black_rot/
#   ...
#
# A folder can also contain nested class directories; discovery below searches recursively.

DATA_ROOT = "/content/PlantVillage"
OUTPUT_DIR = "/content/FedQLeaf_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEMO_MODE = True

# Manuscript configuration
IMAGE_SIZE = 224
NUM_CLIENTS = 12
DIRICHLET_ALPHA = 0.3
BATCH_SIZE = 32
LOCAL_EPOCHS = 1 if DEMO_MODE else 5
ROUNDS = 3 if DEMO_MODE else 100
LR = 3e-4
WEIGHT_DECAY = 1e-4

# Architecture
BOTTLENECK_DIM = 128
NUM_EXPERTS = 4
TOP_K_EXPERTS = 2
NUM_QUBITS = 4
Q_DEPTH = 3
NUM_HEADS = 3

# Early-exit configuration from the manuscript
BASE_ENTROPY_THRESHOLDS = [0.15, 0.25, 0.35]
GATE_COUPLING = 0.2
DEPTH_WEIGHTS = [0.2, 0.3, 0.5]

# Practical data cap: None means use all images.
# Keep None for the full dataset; a small integer makes notebook smoke-testing fast.
MAX_IMAGES = 12000 if DEMO_MODE else None

# Optional cross-dataset folders. They are not required for training.
CROSS_DATASETS = {
    "PlantDoc": "/content/PlantDoc",
    "PlantCity": "/content/PlantCity",
    "MultiCrop": "/content/MultiCrop"
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({
    "DEMO_MODE": DEMO_MODE,
    "rounds": ROUNDS,
    "local_epochs": LOCAL_EPOCHS,
    "clients": NUM_CLIENTS,
    "device": str(DEVICE)
})

In [ ]:
# 3. Dataset discovery
# Expected format: class folders containing JPG/PNG/JPEG/BMP/TIF images.

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def discover_images(root, max_images=None):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(
            f"DATA_ROOT does not exist: {root}\n"
            "Upload/mount the dataset in Colab and update DATA_ROOT."
        )
    rows = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            rel = p.relative_to(root)
            # First directory below root is treated as the class when available.
            parts = rel.parts
            label = parts[0] if len(parts) > 1 else p.parent.name
            rows.append((str(p), label))
    if not rows:
        raise RuntimeError(f"No supported images found under {root}")
    df = pd.DataFrame(rows, columns=["path", "label"])
    if max_images is not None and len(df) > max_images:
        # Stratified cap so small classes are retained.
        sampled = []
        per_class = max(1, max_images // df["label"].nunique())
        for lab, g in df.groupby("label"):
            sampled.append(g.sample(min(len(g), per_class), random_state=SEED))
        df = pd.concat(sampled).sample(frac=1, random_state=SEED).reset_index(drop=True)
        if len(df) > max_images:
            df = df.sample(max_images, random_state=SEED).reset_index(drop=True)
    labels = sorted(df["label"].unique())
    label_to_idx = {x:i for i,x in enumerate(labels)}
    df["target"] = df["label"].map(label_to_idx)
    return df, labels

df, CLASS_NAMES = discover_images(DATA_ROOT, MAX_IMAGES)
NUM_CLASSES = len(CLASS_NAMES)

print(f"Images: {len(df):,}")
print(f"Classes: {NUM_CLASSES}")
print("Class distribution:")
display(df["label"].value_counts().head(30))

if NUM_CLASSES < 2:
    raise ValueError("At least two disease/health classes are required.")

In [ ]:
# 4. Group-aware stratified train/validation/test split
# A lightweight perceptual signature is used to avoid exact duplicate files crossing splits.
# For very large datasets, the full manuscript pHash/augmentation-variant audit can be added
# as a separate preprocessing pass.

def file_signature(path):
    try:
        im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if im is None:
            return None
        im = cv2.resize(im, (32, 32), interpolation=cv2.INTER_AREA)
        return hashlib.sha1(im.tobytes()).hexdigest()
    except Exception:
        return None

df["signature"] = [file_signature(p) for p in tqdm(df["path"], desc="Hashing")]
df = df[df["signature"].notna()].drop_duplicates("signature").reset_index(drop=True)

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["target"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=2/3, stratify=temp_df["target"], random_state=SEED
)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part.to_csv(f"{OUTPUT_DIR}/{name}_manifest.csv", index=False)
    print(name, len(part), "images")

print("\nClass count in train:")
display(train_df["label"].value_counts().sort_index().to_frame("count").head(40))

In [ ]:
# 5. Client formation — FedNonIID-Leaf style Dirichlet partitioning
# The supplied manuscript uses α=0.3 and 12 clients. Each class is distributed across clients
# using a Dirichlet draw, then the resulting client class distribution is measured with KL divergence.

def dirichlet_partition(frame, n_clients, alpha, seed=42):
    rng = np.random.default_rng(seed)
    client_indices = [[] for _ in range(n_clients)]
    for cls in sorted(frame["target"].unique()):
        idx = frame.index[frame["target"] == cls].to_numpy()
        rng.shuffle(idx)
        proportions = rng.dirichlet(np.repeat(alpha, n_clients))
        cuts = (np.cumsum(proportions) * len(idx)).astype(int)[:-1]
        splits = np.split(idx, cuts)
        for c, split in enumerate(splits):
            client_indices[c].extend(split.tolist())
    # Ensure no empty client
    for c in range(n_clients):
        if len(client_indices[c]) == 0:
            donor = max(range(n_clients), key=lambda x: len(client_indices[x]))
            client_indices[c].append(client_indices[donor].pop())
    return client_indices

client_global_indices = dirichlet_partition(train_df, NUM_CLIENTS, DIRICHLET_ALPHA, SEED)

global_counts = np.bincount(train_df["target"].to_numpy(), minlength=NUM_CLASSES).astype(float)
global_p = global_counts / global_counts.sum()

client_stats = []
for cid, idx in enumerate(client_global_indices):
    labs = train_df.loc[idx, "target"].to_numpy()
    counts = np.bincount(labs, minlength=NUM_CLASSES).astype(float)
    p = counts / max(counts.sum(), 1)
    kl = np.sum(np.where(p > 0, p * np.log((p + 1e-12)/(global_p + 1e-12)), 0))
    client_stats.append({
        "client": cid,
        "samples": len(idx),
        "KL_to_global": float(kl),
        "classes_present": int((counts > 0).sum())
    })

client_stats_df = pd.DataFrame(client_stats)
display(client_stats_df)
client_stats_df.to_csv(f"{OUTPUT_DIR}/client_heterogeneity.csv", index=False)

plt.figure(figsize=(10,4))
plt.bar(client_stats_df["client"].astype(str), client_stats_df["samples"])
plt.xlabel("Client"); plt.ylabel("Training samples"); plt.title("FedNonIID-Leaf client sizes")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/client_sizes.png", dpi=220); plt.show()

In [ ]:
# 6. Local preprocessing — GrabCut + CLAHE + normalization
# The manuscript specifies client-local foreground extraction followed by adaptive histogram equalization.

def preprocess_leaf(path, size=224):
    bgr = cv2.imread(path)
    if bgr is None:
        raise ValueError(f"Cannot read image: {path}")
    bgr = cv2.resize(bgr, (size, size), interpolation=cv2.INTER_AREA)

    # GrabCut initialized with a safe border rectangle.
    mask = np.zeros(bgr.shape[:2], np.uint8)
    rect = (max(2, size//16), max(2, size//16),
            size - 2*max(2, size//16), size - 2*max(2, size//16))
    bgd = np.zeros((1,65), np.float64)
    fgd = np.zeros((1,65), np.float64)
    try:
        cv2.grabCut(bgr, mask, rect, bgd, fgd, 2, cv2.GC_INIT_WITH_RECT)
        fg = np.where((mask == 2) | (mask == 0), 0, 1).astype(np.uint8)
        # If GrabCut fails to retain enough foreground, keep the original image.
        if fg.mean() < 0.05 or fg.mean() > 0.98:
            fg[:] = 1
    except Exception:
        fg = np.ones(mask.shape, np.uint8)

    bgr = bgr * fg[..., None]

    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    rgb = cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2RGB)
    return rgb

class LeafDataset(Dataset):
    def __init__(self, frame, train=False):
        self.frame = frame.reset_index(drop=True)
        self.train = train
        self.aug = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(p=0.15),
            transforms.RandomRotation(12),
            transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.12),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
        ])
        self.eval_tf = transforms.Compose([
            transforms.ToPILImage(),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
        ])

    def __len__(self): return len(self.frame)

    def __getitem__(self, i):
        row = self.frame.iloc[i]
        try:
            img = preprocess_leaf(row.path, IMAGE_SIZE)
        except Exception:
            img = cv2.cvtColor(cv2.imread(row.path), cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        x = self.aug(img) if self.train else self.eval_tf(img)
        return x, int(row.target)

train_ds = LeafDataset(train_df, train=True)
val_ds = LeafDataset(val_df, train=False)
test_ds = LeafDataset(test_df, train=False)

x0, y0 = train_ds[0]
print("Example tensor:", tuple(x0.shape), "label:", CLASS_NAMES[y0])

In [ ]:
# 7. LeafFreq-QNet
# This is a compact, runnable implementation of the manuscript's learnable frequency/spatial
# representation. The wavelet-like filters are trainable depthwise convolutions initialized from
# Haar filters. The spatial branch uses additive attention, followed by SF-Gate and top-k MoE.

class LearnableWavelet(nn.Module):
    def __init__(self, channels=3):
        super().__init__()
        h = torch.tensor([1., 1.]) / math.sqrt(2)
        g = torch.tensor([1., -1.]) / math.sqrt(2)
        ll = torch.outer(h,h); lh = torch.outer(h,g); hl = torch.outer(g,h); hh = torch.outer(g,g)
        bank = torch.stack([ll, lh, hl, hh]).unsqueeze(1).repeat(channels,1,1,1)
        self.filters = nn.Parameter(bank)
        self.channels = channels

    def forward(self, x):
        w = F.conv2d(x, self.filters, stride=2, padding=1, groups=self.channels)
        b,c4,h,w2 = w.shape
        w = w.view(b, self.channels, 4, h, w2)
        low = w[:,:,0].mean(dim=1, keepdim=True)
        high = w[:,:,1:].abs().mean(dim=(1,2), keepdim=True)
        return low, high

class AdditiveAttentionPool(nn.Module):
    def __init__(self, in_dim, hidden=128):
        super().__init__()
        self.q = nn.Linear(in_dim, hidden)
        self.v = nn.Linear(in_dim, hidden)
        self.score = nn.Linear(hidden, 1)

    def forward(self, tokens):
        q = torch.tanh(self.q(tokens))
        v = self.v(tokens)
        a = torch.softmax(self.score(q).squeeze(-1), dim=1)
        pooled = torch.sum(v * a.unsqueeze(-1), dim=1)
        return pooled, a

class LeafFreqQNet(nn.Module):
    def __init__(self, bottleneck=128, experts=4, topk=2):
        super().__init__()
        self.topk = topk
        self.wave = LearnableWavelet(3)

        self.spatial_stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 96, 3, stride=2, padding=1), nn.BatchNorm2d(96), nn.GELU()
        )
        self.spatial_pool = AdditiveAttentionPool(96, 128)

        self.low_proj = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(1,128), nn.GELU())
        self.high_proj = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(1,128), nn.GELU())

        self.gate = nn.Sequential(nn.Linear(384,128), nn.GELU(), nn.Dropout(0.1), nn.Linear(128,3))
        self.expert_router = nn.Linear(131, experts)

        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(128,256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256,128))
            for _ in range(experts)
        ])
        self.bottleneck = nn.Linear(128, bottleneck)

    def forward(self, x):
        low, high = self.wave(x)
        spatial = self.spatial_stem(x)
        spatial_vec, attn = self.spatial_pool(spatial.flatten(2).transpose(1,2))
        low_vec = self.low_proj(low)
        high_vec = self.high_proj(high)

        cat = torch.cat([low_vec, high_vec, spatial_vec], dim=1)
        gates = torch.softmax(self.gate(cat), dim=1)

        route_in = torch.cat([torch.cat([low_vec, high_vec, spatial_vec], dim=1)[:,:128], gates], dim=1)
        logits = self.expert_router(route_in)
        topv, topi = torch.topk(logits, self.topk, dim=1)
        weights = torch.softmax(topv, dim=1)

        expert_out = torch.stack([e(spatial_vec) for e in self.experts], dim=1)
        selected = torch.gather(expert_out, 1, topi.unsqueeze(-1).expand(-1,-1,128))
        moe = torch.sum(selected * weights.unsqueeze(-1), dim=1)

        z = F.gelu(self.bottleneck(moe))
        return z, {"sf_gate": gates, "spatial_attention": attn, "expert_logits": logits}

# Smoke test
tmp_model = LeafFreqQNet().to(DEVICE)
with torch.no_grad():
    z,_ = tmp_model(torch.randn(2,3,IMAGE_SIZE,IMAGE_SIZE,device=DEVICE))
print("LeafFreq-QNet output:", tuple(z.shape))
del tmp_model

In [ ]:
# 8. Quantum Interaction Fusion (QIF)
# Four-qubit angle encoding + hardware-efficient variational circuit.
# PennyLane's default.qubit simulator keeps the notebook runnable without quantum hardware.

dev = qml.device("default.qubit", wires=NUM_QUBITS)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    for i in range(NUM_QUBITS):
        qml.RY(inputs[i], wires=i)
        qml.RZ(inputs[i], wires=i)
    for d in range(Q_DEPTH):
        for i in range(NUM_QUBITS):
            qml.RY(weights[d,i,0], wires=i)
            qml.RZ(weights[d,i,1], wires=i)
        for i in range(NUM_QUBITS-1):
            qml.CNOT(wires=[i, i+1])
    return [qml.expval(qml.PauliZ(i)) for i in range(NUM_QUBITS)]

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(0.02 * torch.randn(Q_DEPTH, NUM_QUBITS, 2))

    def forward(self, x):
        # PennyLane processes one sample at a time here for broad compatibility.
        outs = []
        for row in x:
            angles = torch.tanh(row[:NUM_QUBITS]) * math.pi
            q = quantum_circuit(angles, self.weights)
            outs.append(torch.stack(q))
        return torch.stack(outs)

class QIF(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.to_qubits = nn.Linear(dim, NUM_QUBITS)
        self.q = QuantumLayer()
        self.q_to_dim = nn.Linear(NUM_QUBITS, dim)
        self.gate = nn.Sequential(nn.Linear(dim*2, dim), nn.Sigmoid())
        self.fuse = nn.Sequential(nn.Linear(dim*2, dim), nn.GELU(), nn.Dropout(0.1))

    def forward(self, x):
        qout = self.q(x)
        qproj = self.q_to_dim(qout)
        gate = self.gate(torch.cat([x, qproj], dim=1))
        fused = self.fuse(torch.cat([x, gate*qproj], dim=1))
        return fused, {"quantum": qproj, "interaction_gate": gate}

# Quantum smoke test
qif_test = QIF().to(DEVICE)
with torch.no_grad():
    qz,_ = qif_test(torch.randn(2,128,device=DEVICE))
print("QIF output:", tuple(qz.shape))
del qif_test

In [ ]:
# 9. LeafQ-DiagnoseNet — three early-exit heads + gate-aware entropy criterion

class DiagnoseNet(nn.Module):
    def __init__(self, dim=128, n_classes=2):
        super().__init__()
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(dim,64), nn.GELU(), nn.Linear(64,n_classes)),
            nn.Sequential(nn.Linear(dim,96), nn.GELU(), nn.Linear(96,n_classes)),
            nn.Sequential(nn.Linear(dim,128), nn.GELU(), nn.Linear(128,n_classes))
        ])

    def forward(self, x):
        return [h(x) for h in self.heads]

def entropy_from_logits(logits):
    p = torch.softmax(logits, dim=-1)
    return -(p * torch.log(p.clamp_min(1e-8))).sum(dim=-1)

def choose_exit(head_logits, gate, thresholds=BASE_ENTROPY_THRESHOLDS, coupling=GATE_COUPLING):
    # Larger classical reliance (gate value) tightens the threshold, as described in the manuscript.
    # Use mean gate activation per sample.
    classical_share = gate.mean(dim=1)
    chosen = torch.full((head_logits[0].shape[0],), len(head_logits)-1,
                        device=head_logits[0].device, dtype=torch.long)
    for d, logits in enumerate(head_logits):
        ent = entropy_from_logits(logits)
        # Convert gate to a threshold tightening term.
        thr = thresholds[d] * (1.0 - coupling * classical_share)
        confident = ent <= thr
        undecided = chosen == len(head_logits)-1
        chosen = torch.where(undecided & confident,
                             torch.full_like(chosen, d), chosen)
    return chosen

def gather_exit_logits(head_logits, chosen):
    out = torch.zeros_like(head_logits[-1])
    for d, logits in enumerate(head_logits):
        m = chosen == d
        if m.any():
            out[m] = logits[m]
    return out

class FedQLeafNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.feature = LeafFreqQNet(BOTTLENECK_DIM, NUM_EXPERTS, TOP_K_EXPERTS)
        self.qif = QIF(BOTTLENECK_DIM)
        self.classifier = DiagnoseNet(BOTTLENECK_DIM, n_classes)

    def forward(self, x, use_early_exit=True):
        z, fmeta = self.feature(x)
        fused, qmeta = self.qif(z)
        heads = self.classifier(fused)
        if use_early_exit:
            chosen = choose_exit(heads, qmeta["interaction_gate"])
            final_logits = gather_exit_logits(heads, chosen)
        else:
            chosen = torch.full((x.size(0),), 2, device=x.device, dtype=torch.long)
            final_logits = heads[-1]
        return {
            "heads": heads,
            "logits": final_logits,
            "exit": chosen,
            "feature_meta": fmeta,
            "qif_meta": qmeta
        }

model = FedQLeafNet(NUM_CLASSES).to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# 10. Loss, fake quantization and local client training
# The loss follows the manuscript's depth-weighted multi-head objective plus a quantum-preservation term.

def multihead_loss(out, y):
    losses = [F.cross_entropy(logits, y) for logits in out["heads"]]
    total = sum(w*l for w,l in zip(DEPTH_WEIGHTS, losses))

    gate_mean = out["qif_meta"]["interaction_gate"].mean()
    preservation_penalty = F.relu(0.15 - gate_mean)
    total = total + 0.01 * preservation_penalty
    return total

def fake_quantize_tensor(x, bits=8):
    qmax = (2**(bits-1))-1
    scale = x.detach().abs().max().clamp_min(1e-8) / qmax
    q = torch.clamp(torch.round(x/scale), -qmax, qmax) * scale
    return x + (q - x).detach()

def apply_fake_quantization(module, bits=8):
    with torch.no_grad():
        for name, p in module.named_parameters():
            if p.ndim >= 2:
                p.copy_(fake_quantize_tensor(p, bits))

def make_loader(frame, indices=None, train=False, batch_size=BATCH_SIZE):
    if indices is not None:
        sub = frame.loc[indices]
    else:
        sub = frame
    ds = LeafDataset(sub, train=train)
    return DataLoader(ds, batch_size=batch_size, shuffle=train,
                      num_workers=2, pin_memory=torch.cuda.is_available())

def local_train(global_model, client_frame, epochs=1):
    local = copy.deepcopy(global_model).to(DEVICE)
    local.train()
    loader = make_loader(train_df, client_frame, train=True)
    opt = torch.optim.AdamW(local.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    losses = []
    for ep in range(epochs):
        ep_loss = 0.0
        n = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            out = local(xb, use_early_exit=False)
            loss = multihead_loss(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(local.parameters(), 5.0)
            opt.step()
            ep_loss += loss.item() * len(yb)
            n += len(yb)
        losses.append(ep_loss/max(n,1))
    # QAT-style weight perturbation before communication.
    apply_fake_quantization(local, bits=8)
    return local.cpu(), float(np.mean(losses))

In [ ]:
# 11. Federated weighted aggregation

def weighted_fedavg(states, weights):
    total = float(sum(weights))
    agg = {}
    for key in states[0].keys():
        ref = states[0][key]
        if torch.is_floating_point(ref):
            agg[key] = sum(st[key] * (w/total) for st,w in zip(states,weights))
        else:
            agg[key] = ref
    return agg

def evaluate(model, dataset, batch_size=64):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=torch.cuda.is_available())
    ys, ps, probs, exits = [], [], [], []
    total_loss = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out = model(xb, use_early_exit=True)
            loss = multihead_loss(out, yb)
            p = torch.softmax(out["logits"], dim=1)
            pred = p.argmax(1)
            total_loss += loss.item()*len(yb); n += len(yb)
            ys.extend(yb.cpu().numpy()); ps.extend(pred.cpu().numpy())
            probs.extend(p.cpu().numpy()); exits.extend(out["exit"].cpu().numpy())
    ys, ps, probs = np.array(ys), np.array(ps), np.array(probs)
    avg = total_loss/max(n,1)
    acc = accuracy_score(ys, ps)
    prec, rec, f1, _ = precision_recall_fscore_support(
        ys, ps, average="weighted", zero_division=0
    )
    try:
        auc = roc_auc_score(ys, probs, multi_class="ovr", average="weighted")
    except Exception:
        auc = float("nan")
    return {
        "loss": avg, "accuracy": acc, "precision": prec,
        "recall": rec, "f1": f1, "roc_auc": auc,
        "y_true": ys, "y_pred": ps, "probs": probs, "exits": np.array(exits)
    }

history = []
global_model = FedQLeafNet(NUM_CLASSES).to(DEVICE)

for rnd in range(1, ROUNDS+1):
    start = time.time()
    client_states, client_weights, client_losses = [], [], []

    # In DEMO_MODE use all clients but only one local epoch/round.
    for cid in range(NUM_CLIENTS):
        local_model, loss = local_train(global_model, client_global_indices[cid], LOCAL_EPOCHS)
        client_states.append(local_model.state_dict())
        client_weights.append(len(client_global_indices[cid]))
        client_losses.append(loss)

    new_state = weighted_fedavg(client_states, client_weights)
    global_model.load_state_dict(new_state)
    global_model.to(DEVICE)

    val = evaluate(global_model, val_ds)
    row = {
        "round": rnd,
        "train_loss": float(np.average(client_losses, weights=client_weights)),
        "val_loss": val["loss"],
        "val_accuracy": val["accuracy"],
        "val_precision": val["precision"],
        "val_recall": val["recall"],
        "val_f1": val["f1"],
        "val_auc": val["roc_auc"],
        "time_sec": time.time()-start
    }
    history.append(row)
    print(
        f"Round {rnd:03d}/{ROUNDS} | "
        f"loss={row['train_loss']:.4f} | val_acc={row['val_accuracy']*100:.2f}% | "
        f"val_f1={row['val_f1']*100:.2f}% | {row['time_sec']:.1f}s"
    )

history_df = pd.DataFrame(history)
history_df.to_csv(f"{OUTPUT_DIR}/federated_history.csv", index=False)
torch.save(global_model.state_dict(), f"{OUTPUT_DIR}/FedQLeaf_global.pt")

In [ ]:
# 12. Training curves

fig = plt.figure(figsize=(10,4))
plt.plot(history_df["round"], history_df["train_loss"], marker="o", label="Train loss")
plt.plot(history_df["round"], history_df["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Federated round"); plt.ylabel("Loss"); plt.title("FedQLeaf-Net training loss")
plt.legend(); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png", dpi=220)
plt.show()

fig = plt.figure(figsize=(10,4))
plt.plot(history_df["round"], history_df["val_accuracy"]*100, marker="o", label="Accuracy")
plt.plot(history_df["round"], history_df["val_f1"]*100, marker="o", label="F1")
plt.xlabel("Federated round"); plt.ylabel("Percentage")
plt.title("Federated validation performance")
plt.legend(); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/validation_metrics.png", dpi=220)
plt.show()

In [ ]:
# 13. Final test evaluation

test_result = evaluate(global_model, test_ds)

metric_table = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Value (%)": [
        test_result["accuracy"]*100,
        test_result["precision"]*100,
        test_result["recall"]*100,
        test_result["f1"]*100,
        test_result["roc_auc"]*100 if np.isfinite(test_result["roc_auc"]) else np.nan
    ]
})
display(metric_table)
metric_table.to_csv(f"{OUTPUT_DIR}/final_metrics.csv", index=False)

print("\nClassification report:")
print(classification_report(
    test_result["y_true"], test_result["y_pred"],
    target_names=CLASS_NAMES, zero_division=0
))

cm = confusion_matrix(test_result["y_true"], test_result["y_pred"])
plt.figure(figsize=(max(7, len(CLASS_NAMES)*0.55), max(6, len(CLASS_NAMES)*0.45)))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("FedQLeaf-Net confusion matrix")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", dpi=240); plt.show()

In [ ]:
# 14. Early-exit analysis

exit_counts = np.bincount(test_result["exits"], minlength=NUM_HEADS)
exit_pct = 100 * exit_counts / max(exit_counts.sum(), 1)

exit_df = pd.DataFrame({
    "Exit head": ["Head 1", "Head 2", "Head 3"],
    "Samples": exit_counts,
    "Percentage": exit_pct
})
display(exit_df)
exit_df.to_csv(f"{OUTPUT_DIR}/early_exit_usage.csv", index=False)

plt.figure(figsize=(7,4))
plt.bar(exit_df["Exit head"], exit_df["Percentage"])
plt.ylabel("Samples (%)"); plt.title("Early-exit head utilization")
for i,v in enumerate(exit_pct):
    plt.text(i, v+0.5, f"{v:.1f}%", ha="center")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/early_exit_usage.png", dpi=220); plt.show()

In [ ]:
# 15. Post-training mixed-precision weight quantization
# This creates a separate state dictionary for edge deployment. It does not claim integer-only
# runtime unless an actual backend conversion is performed.

def quantize_state_dict(state_dict, bits=8):
    out = {}
    qmax = 2**(bits-1)-1
    for k,v in state_dict.items():
        if torch.is_floating_point(v) and v.ndim >= 2:
            scale = v.abs().max().clamp_min(1e-8)/qmax
            out[k] = torch.round(v/scale).clamp(-qmax,qmax).to(torch.int8)
            out[k+"_scale"] = scale.cpu()
        else:
            out[k] = v
    return out

qstate = quantize_state_dict(global_model.state_dict(), bits=8)
torch.save(qstate, f"{OUTPUT_DIR}/FedQLeaf_8bit_state.pt")

fp32_bytes = sum(v.numel()*v.element_size() for v in global_model.state_dict().values())
int8_bytes = 0
for k,v in qstate.items():
    if torch.is_tensor(v):
        int8_bytes += v.numel()*v.element_size()

print(f"Approx. FP32 state size: {fp32_bytes/1024**2:.2f} MB")
print(f"Approx. quantized state size: {int8_bytes/1024**2:.2f} MB")
print(f"Approx. reduction: {(1-int8_bytes/fp32_bytes)*100:.2f}%")

In [ ]:
# 16. Communication-cost estimate
# The manuscript's communication analysis separates classical parameters and the small quantum
# circuit parameter set. This notebook reports an empirical FP32-equivalent upper bound and
# an 8-bit classifier-style approximation.

num_params = sum(p.numel() for p in global_model.parameters())
quantizable = sum(p.numel() for n,p in global_model.named_parameters() if p.ndim >= 2)
quant_bits = 8
quantized_upload_bits = quantizable*quant_bits + (num_params-quantizable)*32

comm_per_round_mb = NUM_CLIENTS * quantized_upload_bits / 8 / 1024**2
comm_total_mb = comm_per_round_mb * ROUNDS

comm_df = pd.DataFrame({
    "Quantity": [
        "Total parameters", "Quantizable parameters",
        "Estimated client upload per round (MB)",
        "Estimated total client upload (MB)"
    ],
    "Value": [num_params, quantizable, comm_per_round_mb, comm_total_mb]
})
display(comm_df)
comm_df.to_csv(f"{OUTPUT_DIR}/communication_estimate.csv", index=False)

In [ ]:
# 17. Single-image inference
# Set SINGLE_IMAGE to any leaf image path. The result reports predicted class, confidence,
# selected early-exit depth and the quantum interaction-gate mean.

SINGLE_IMAGE = None  # e.g. "/content/PlantVillage/Tomato___Late_blight/image.jpg"

def predict_one(model, image_path):
    model.eval()
    frame = pd.DataFrame({"path":[image_path], "label":["inference"], "target":[0]})
    ds = LeafDataset(frame, train=False)
    x,_ = ds[0]
    with torch.no_grad():
        out = model(x.unsqueeze(0).to(DEVICE), use_early_exit=True)
        p = torch.softmax(out["logits"], dim=1)[0].cpu().numpy()
        pred = int(p.argmax())
        conf = float(p[pred])
        exit_id = int(out["exit"][0].item()) + 1
        gate = float(out["qif_meta"]["interaction_gate"].mean().item())
    return {
        "prediction": CLASS_NAMES[pred],
        "confidence_percent": conf*100,
        "selected_exit_head": exit_id,
        "quantum_classical_gate_mean": gate
    }

if SINGLE_IMAGE is not None:
    print(predict_one(global_model, SINGLE_IMAGE))
else:
    print("Set SINGLE_IMAGE to an image path to run inference.")

In [ ]:
# 18. Save a compact experiment summary

summary = {
    "dataset_root": DATA_ROOT,
    "images_after_dedup": int(len(df)),
    "classes": CLASS_NAMES,
    "num_classes": NUM_CLASSES,
    "clients": NUM_CLIENTS,
    "dirichlet_alpha": DIRICHLET_ALPHA,
    "rounds": ROUNDS,
    "local_epochs": LOCAL_EPOCHS,
    "image_size": IMAGE_SIZE,
    "bottleneck_dim": BOTTLENECK_DIM,
    "qubits": NUM_QUBITS,
    "quantization_bits": 8,
    "test_accuracy": float(test_result["accuracy"]),
    "test_precision": float(test_result["precision"]),
    "test_recall": float(test_result["recall"]),
    "test_f1": float(test_result["f1"]),
    "test_roc_auc": float(test_result["roc_auc"]) if np.isfinite(test_result["roc_auc"]) else None
}
with open(f"{OUTPUT_DIR}/experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nAll outputs saved under:", OUTPUT_DIR)

## Implementation notes

- The manuscript describes PlantVillage, PlantDoc, PlantCity and Multi-Crop Disease Dataset as the four evaluation sources; this notebook trains on the folder specified by `DATA_ROOT` and keeps optional cross-dataset paths configurable. fileciteturn0file0L247-L260
- The federated partition uses the manuscript's Dirichlet non-IID setting (`α=0.3`, 12 clients). fileciteturn0file0L325-L345
- The preprocessing follows the stated client-local foreground extraction and CLAHE sequence. fileciteturn0file0L359-L378
- LeafFreq-QNet uses the manuscript's 128-D bottleneck, four experts and top-2 routing. fileciteturn0file0L445-L455
- QIF uses the specified 4-qubit budget, depth 3 and 128-D fusion representation. fileciteturn0file0L501-L508
- LeafQ-DiagnoseNet contains three heads with the stated entropy thresholds, coupling coefficient and depth weighting. fileciteturn0file0L559-L566
- The supplied manuscript says Flower, PennyLane, OpenCV and PyTorch are used for the experimental implementation. This notebook uses PennyLane/OpenCV/PyTorch directly and performs the federated weighted aggregation in-process, which avoids the extra complexity of launching a Flower server/client process inside a single Colab notebook. fileciteturn0file0L663-L671
- `DEMO_MODE=True` is intentionally practical for a first run. Change it to `False` to use the manuscript's 100 rounds and 5 local epochs. Actual metrics depend on the dataset and hardware and are computed rather than hard-coded.